In [1]:
import glob
import importlib.util
import os
import site
import tempfile
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in os.sys.path:
    os.sys.path.insert(0, str(PROJECT_ROOT))

def bootstrap_repo_venv_site_packages(project_root: Path):
    version_tag = f"python{os.sys.version_info.major}.{os.sys.version_info.minor}"
    candidates = [
        project_root / ".venv" / "Lib" / "site-packages",
        project_root / ".venv" / "lib" / version_tag / "site-packages",
    ]
    for candidate in candidates:
        if candidate.exists():
            site.addsitedir(str(candidate))
            return candidate
    return None

required_packages = ("matplotlib", "mplfinance", "numpy", "pandas", "seaborn", "ipywidgets")
missing = [package_name for package_name in required_packages if importlib.util.find_spec(package_name) is None]
venv_site_packages = None
if missing:
    venv_site_packages = bootstrap_repo_venv_site_packages(PROJECT_ROOT)
    missing = [package_name for package_name in required_packages if importlib.util.find_spec(package_name) is None]
if missing:
    install_target = f'{PROJECT_ROOT}[analysis,storage,websocket]'
    raise ModuleNotFoundError(
        "Notebook dependencies are missing in the current kernel "
        f'({os.sys.executable}). Missing: {", ".join(missing)}. '
        f'Run `%pip install -e \"{install_target}\"` in this notebook, '
        'or switch VS Code/Jupyter to the "mykalshi (.venv)" kernel.'
    )
if venv_site_packages is not None and ".venv" not in os.sys.executable:
    print(
        "Notebook is running outside the repo venv; "
        f"using packages from {venv_site_packages}"
    )

from mykalshi import events, formatting, market, routing, trading, communications, exchange
from mykalshi.config import KalshiConfig
from mykalshi.exceptions import KalshiHTTPError
from mykalshi.recorder import MarketLOBRecorder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mplfinance as mpf
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from IPython.display import clear_output, display

try:
    from ipywidgets import interact, IntSlider, fixed
    IPYWIDGETS_AVAILABLE = True
except ModuleNotFoundError:
    IPYWIDGETS_AVAILABLE = False

sns.set_theme(style="darkgrid")
%config InlineBackend.figure_format = 'retina'

CONFIG = KalshiConfig.from_env()
AUTH_AVAILABLE = bool(CONFIG.api_key_id and CONFIG.private_key_path)
PRESIDENTIAL_EVENT_TICKER = "PRES-2024"
WEATHER_SERIES = {
    'KXHIGHUS': 'High temp in United States',
    'KXHIGHAUS': 'Highest temperature in Austin',
    'KXHIGHCHI': 'Highest temperature in Chicago',
    'KXHIGHDEN': 'Highest temperature in Denver',
    'KXHIGHHOU': 'Highest temperature in Houston',
    'KXHIGHLAX': 'Highest temperature in Los Angeles',
    'KXHIGHMIA': 'Highest temperature in Miami',
    'KXHIGHNY': 'Highest temperature in NYC',
    'KXHIGHPHIL': 'Highest temperature in Philadelphia',
}

def latest_market_snapshot_path() -> Path | None:
    files = sorted(PROJECT_ROOT.glob("all_markets_*.csv"), key=lambda path: path.stat().st_mtime)
    return files[-1] if files else None

def snapshot_path_for_refresh() -> Path:
    existing = latest_market_snapshot_path()
    if existing is not None:
        return existing
    return PROJECT_ROOT / f"all_markets_{datetime.now(timezone.utc).strftime('%Y-%m-%d-%H-%M-%S')}.csv"

MARKET_HISTORY_CACHE = {}
MARKET_SNAPSHOT_COLUMNS = {
    "ticker",
    "event_ticker",
    "market_type",
    "title",
    "subtitle",
    "yes_sub_title",
    "no_sub_title",
    "open_time",
    "close_time",
    "status",
    "last_price",
    "yes_bid",
    "yes_ask",
    "no_bid",
    "no_ask",
    "volume",
    "volume_24h",
    "open_interest",
    "liquidity",
    "last_price_dollars",
    "yes_bid_dollars",
    "yes_ask_dollars",
    "no_bid_dollars",
    "no_ask_dollars",
    "volume_fp",
    "volume_24h_fp",
    "open_interest_fp",
    "liquidity_dollars",
}
MARKET_SNAPSHOT_DTYPES = {
    "ticker": "string",
    "event_ticker": "string",
    "market_type": "category",
    "title": "string",
    "subtitle": "string",
    "yes_sub_title": "string",
    "no_sub_title": "string",
    "open_time": "string",
    "close_time": "string",
    "status": "category",
}

def retry_on_429(label: str, func, *, max_retries: int = 4, base_backoff_seconds: float = 1.0):
    for attempt in range(1, max_retries + 1):
        try:
            return func()
        except KalshiHTTPError as exc:
            if exc.status_code != 429 or attempt >= max_retries:
                raise
            sleep_seconds = base_backoff_seconds * attempt
            print(f"Rate limited while loading {label}; retrying in {sleep_seconds:.1f}s...")
            time.sleep(sleep_seconds)

def normalize_market_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if frame.empty:
        return frame
    for target, source in {
        "last_price": "last_price_dollars",
        "yes_bid": "yes_bid_dollars",
        "yes_ask": "yes_ask_dollars",
        "no_bid": "no_bid_dollars",
        "no_ask": "no_ask_dollars",
    }.items():
        if target not in frame.columns and source in frame.columns:
            frame[target] = (pd.to_numeric(frame[source], errors="coerce") * 100).round()
    for target, source in {
        "volume": "volume_fp",
        "volume_24h": "volume_24h_fp",
        "open_interest": "open_interest_fp",
        "liquidity": "liquidity_dollars",
    }.items():
        if target not in frame.columns and source in frame.columns:
            frame[target] = pd.to_numeric(frame[source], errors="coerce")
    return frame

def load_market_snapshot() -> pd.DataFrame:
    snapshot_path = snapshot_path_for_refresh()
    sync_summary = market.sync_market_snapshot_csv(snapshot_path)
    print(
        f"Loaded market snapshot from {snapshot_path.name} "
        f"(mode={sync_summary['mode']}, delta_count={sync_summary['delta_count']})"
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=pd.errors.DtypeWarning)
        frame = pd.read_csv(
            snapshot_path,
            low_memory=False,
            usecols=lambda column: column in MARKET_SNAPSHOT_COLUMNS,
            dtype=MARKET_SNAPSHOT_DTYPES,
        )
    return normalize_market_frame(frame)

def load_open_markets(limit: int = 300) -> pd.DataFrame:
    payload = pd.json_normalize(market.get_all_markets(status="open", batch_size=100, max_items=limit))
    return normalize_market_frame(payload)

def pick_lob_market(open_markets: pd.DataFrame, probe_limit: int = 20) -> str:
    explicit = os.getenv("MYKALSHI_NOTEBOOK_LOB_TICKER")
    if explicit:
        return explicit
    sortable = open_markets.copy()
    if "volume" in sortable.columns:
        sortable = sortable.sort_values(["volume", "ticker"], ascending=[False, True])
    fallback = str(sortable.iloc[0]["ticker"])
    quoted = sortable
    if {"yes_bid", "yes_ask"}.issubset(sortable.columns):
        quoted = sortable[sortable["yes_bid"].notna() & sortable["yes_ask"].notna()]
        if not AUTH_AVAILABLE and not quoted.empty:
            return str(quoted.iloc[0]["ticker"])
    if not AUTH_AVAILABLE:
        return fallback

    best_one_sided = None
    candidates = list(quoted["ticker"].head(probe_limit)) if not quoted.empty else []
    seen = set(str(ticker) for ticker in candidates)
    candidates.extend(
        str(ticker) for ticker in sortable["ticker"].head(probe_limit * 2)
        if str(ticker) not in seen
    )
    for ticker in candidates:
        try:
            orderbook = market.get_market_orderbook(str(ticker)).get("orderbook", {})
        except Exception:
            continue
        yes_levels = orderbook.get("yes", [])
        no_levels = orderbook.get("no", [])
        if yes_levels and no_levels:
            return str(ticker)
        if (yes_levels or no_levels) and best_one_sided is None:
            best_one_sided = str(ticker)

    return best_one_sided or fallback

def load_full_market_cached(
    series_ticker: str,
    ticker: str,
    period_interval: str,
    *,
    start_ts: str | None = None,
    end_ts: str | None = None,
    max_retries: int = 4,
    base_backoff_seconds: float = 1.0,
):
    cache_key = (series_ticker, ticker, period_interval, start_ts, end_ts)
    if cache_key in MARKET_HISTORY_CACHE:
        return MARKET_HISTORY_CACHE[cache_key]

    result = retry_on_429(
        ticker,
        lambda: market.get_full_market(
            series_ticker=series_ticker,
            ticker=ticker,
            period_interval=period_interval,
            start_ts=start_ts,
            end_ts=end_ts,
        ),
        max_retries=max_retries,
        base_backoff_seconds=base_backoff_seconds,
    )
    MARKET_HISTORY_CACHE[cache_key] = result
    return result

def extract_city_from_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.extract(
        r'in\s+(.+?)(?=\s+(?:on|today|tomorrow|yesterday)\b|[?]|$)',
        expand=False,
    ).str.strip()


Notebook is running outside the repo venv; using packages from c:\Users\nicco\OneDrive\Documents\Github\mykalshi\.venv\Lib\site-packages


## 0.0 Preliminary and Brief Analysis

In [ ]:
markets_df = load_market_snapshot()
display(markets_df.head(10))


KalshiHTTPError: GET https://api.elections.kalshi.com/trade-api/v2/markets failed with status 502: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
</body>
</html>


: 

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=markets_df, x='volume', bins=100, kde=False, stat='proportion')
plt.title('Distribution of Finalized Liquid Markets by Volume')
plt.xlabel('Volume')
plt.ylabel('Frequency')
plt.yscale('log')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1e6)}M'))
plt.show()

In [ ]:
bins = [[0, 0.1], [0.1, 10000], [10000, 100000], [100000, 1000000], [1000000, 10000000], [10000000, 50000000], [50000000, 100000000000]]
bins_labels = ["No Volume", "< 10k", "10k-100k", "100k-1M", "1M-10M", "10M-50M", "50M+"]

def categorize_volume(volume):
    for i, (low, high) in enumerate(bins):
        if low <= volume < high:
            return f"{low}-{high}"
    return "Other"

markets_df['volume_category'] = markets_df['volume'].apply(categorize_volume)
plt.figure(figsize=(12, 6)) 
sns.countplot(data=markets_df, x='volume_category', order=sorted(markets_df['volume_category'].unique()))
# Override the x-tick labels to show ranges
plt.xticks(ticks=range(len(bins_labels)), labels=bins_labels, rotation=45)
# Add printed number on top of each bar (and formate them clearly with "," thousands separator)
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                       ha='center', va='bottom', fontsize=10, color='black', rotation=0)
plt.title('Volume Categories of Finalized Liquid Markets')
plt.xlabel('Volume Category')
plt.ylabel('Count')
plt.yscale('log')
plt.show()

In [ ]:
fin_liquid_markets = markets_df[(markets_df['status'] == "finalized") & (markets_df['volume'] > 500000)]
fin_liquid_markets = fin_liquid_markets.reset_index(drop=True)
# fin_liquid_markets.to_csv("FinalizedLiquidMarkets.csv", index=False)
fin_liquid_markets

In [ ]:
open_mkts = load_open_markets(limit=300)
LOB_MARKET_TICKER = pick_lob_market(open_mkts)
display(open_mkts.sort_values(by='volume', ascending=False).head(25))
print(f"Using LOB market: {LOB_MARKET_TICKER}")


## 1.0 Presidential Election

In [ ]:
out = events.event_info(PRESIDENTIAL_EVENT_TICKER)
display(pd.DataFrame([out["event_info"]]))
display(out["markets"].sort_values("volume", ascending=False))


In [ ]:
results = {}

for mkt in out["markets"]["market_ticker"].values:
    cs_df = market.candlesticks_to_df(
        load_full_market_cached(series_ticker="PRES", ticker=mkt, period_interval='h', end_ts='11/10/2024')
    )[['end_period', 'yes_ask_close', 'yes_bid_close', 'volume']]

    cs_df['end_period'] = pd.to_datetime(cs_df['end_period'])
    cs_df['date'] = cs_df['end_period']
    cs_df['midval'] = (cs_df['yes_ask_close'] + cs_df['yes_bid_close']) / 2
    cs_df['ask'] = cs_df['yes_ask_close']
    cs_df['bid'] = cs_df['yes_bid_close']
    cs_df = cs_df.drop(columns=['yes_ask_close', 'yes_bid_close', 'end_period'])
    cs_df.set_index('date', inplace=True)
    title = out["markets"].loc[out["markets"]["market_ticker"] == mkt, "yes_sub_title"].values[0]
    results[title] = cs_df

all_indices = sorted(set().union(*[df.index for df in results.values()]))
global_index = pd.DatetimeIndex(all_indices)
results_df = pd.DataFrame(index=global_index)

for title, df in results.items():
    df = df.groupby(df.index).mean()
    renamed_df = df.rename(columns={
        'midval': f'{title}_midval',
        'ask': f'{title}_ask',
        'bid': f'{title}_bid',
        'volume': f'{title}_volume'
    })
    renamed_df = renamed_df.reindex(global_index)
    results_df = pd.concat([results_df, renamed_df], axis=1)

for col in results_df.columns:
    if '_midval' in col:
        vol_col = col.replace('_midval', '_volume')
        filled_series = results_df[col].copy()
        volume_series = results_df[vol_col]
        for i in range(1, len(filled_series)):
            if pd.isna(filled_series.iloc[i]) and pd.notna(volume_series.iloc[i-1]) and volume_series.iloc[i-1] > 200:
                filled_series.iloc[i] = filled_series.iloc[i - 1]
        results_df[col] = filled_series

results_df['Vol'] = results_df[[col for col in results_df.columns if '_volume' in col]].sum(axis=1)
results_df['Tot'] = results_df[[col for col in results_df.columns if '_midval' in col]].sum(axis=1)
results_df.index.name = 'date'
results_df


In [ ]:
plot_df = results_df.reset_index()

# Identify price columns (all columns except 'date', 'Tot', and 'Vol')
price_columns = [col for col in plot_df.columns if "_midval" in col]

# Create a figure with 3 subplots, making the top two price plots the same size
fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows=4, figsize=(14, 16), 
                                    gridspec_kw={'height_ratios': [2, 2, 2, 1]}, 
                                    sharex=True)

# Plot 1: Individual Prices
for col in price_columns:
    ax1.plot(plot_df['date'], plot_df[col], linewidth=1.5, label=col)
ax1.set_ylabel('Individual Prices')
ax1.set_title('Individual Prices Over Time')
ax1.legend(loc='upper left', ncol=min(4, len(price_columns)))
ax1.grid(True)
plt.setp(ax1.get_xticklabels(), visible=False)

# Plot 2: Bid-Ask Prices
for col in price_columns:
    ax2.plot(plot_df['date'], plot_df[col.replace('_midval', '_ask')] - plot_df[col.replace('_midval', '_bid')], linestyle='--', linewidth=1, label=col.replace('_midval', '_spread'))
    # ax2.plot(plot_df['date'], plot_df[col.replace('_midval', '_bid')], linestyle=':', linewidth=1, label=col.replace('_midval', '_bid'))
ax2.set_ylabel('Bid-Ask Prices')
ax2.set_title('Bid-Ask Prices Over Time')
ax2.legend(loc='upper left', ncol=min(4, len(price_columns) * 2))
ax2.grid(True)
plt.setp(ax2.get_xticklabels(), visible=False)

# selected_outcome = 'Donald Trump'

# ax2.plot(plot_df['date'], plot_df[selected_outcome + '_ask'], linestyle='--', linewidth=1, label=selected_outcome + '_ask')
# ax2.plot(plot_df['date'], plot_df[selected_outcome + '_bid'], linestyle=':', linewidth=1, label=selected_outcome + '_bid')
# ax2.set_ylabel('Bid-Ask Prices')
# ax2.set_title('Bid-Ask Prices Over Time')
# ax2.legend(loc='upper left', ncol=min(4, len(price_columns) * 2))
# ax2.grid(True)
# plt.setp(ax2.get_xticklabels(), visible=False)

# Plot 3: Total Probability
ax3.plot(plot_df['date'], plot_df['Tot'], color='blue', linewidth=1, label='Tot')
ax3.set_ylabel('Total Probability')
ax3.set_title('Total Probability Over Time')
ax3.legend(loc='upper left')
ax3.grid(True)
plt.setp(ax3.get_xticklabels(), visible=False)

# Calculate appropriate width for bars
if plot_df.shape[0] > 1:
    time_diff = (plot_df['date'].iloc[1] - plot_df['date'].iloc[0]).total_seconds()
    width_in_days = (time_diff / (24 * 60 * 60)) * 0.8
else:
    width_in_days = 0.01

# Plot 4: Volume
ax4.plot(plot_df['date'], plot_df['Vol'], color='orange', label='Volume')
ax4.set_ylabel('Volume')
ax4.set_title('Volume Over Time')
ax4.legend(loc='upper left')
ax4.grid(True)

# Format only the bottom x-axis with more compact labels
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
ax4.xaxis.set_major_locator(mdates.DayLocator(interval=7))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45)

# Adjust the x-axis limits to match your data range
date_min = plot_df['date'].min()
date_max = plot_df['date'].max()
ax4.set_xlim(date_min, date_max)

plt.tight_layout()
plt.show()

In [ ]:
pres_djt_data = load_full_market_cached(
    series_ticker="PRES",
    ticker="PRES-2024-DJT",
    end_ts="11/10/2024",
    period_interval='d'
)

pres_kh_data = load_full_market_cached(
    series_ticker="PRES",
    ticker="PRES-2024-KH",
    end_ts="11/10/2024",
    period_interval='d'
)
djt_df = market.candlesticks_to_df(pres_djt_data)
kh_df = market.candlesticks_to_df(pres_kh_data)

djt_candlestick_df = market.build_candlestick(pres_djt_data)
kh_candlestick_df = market.build_candlestick(pres_kh_data)

fig, axes = mpf.plot(
    kh_candlestick_df,
    volume=True,
    figratio=(20, 10),
    figscale=1.8,
    show_nontrading=False,
    mav=3 * 24,
    returnfig=True,
    type='candle',
)
_ = axes[0].axhline(y=50, color='red', linestyle='--', linewidth=1)
_ = axes[0].axvline(x=datetime.strptime('Feb 3 1970 00:00', '%b %d %Y %H:%M'), color='blue', linestyle='--', linewidth=1)
plt.show()

fig, axes = mpf.plot(
    djt_candlestick_df,
    volume=True,
    figratio=(20, 10),
    figscale=1.8,
    show_nontrading=False,
    mav=3 * 24,
    returnfig=True,
    type='candle',
)
_ = axes[0].axhline(y=50, color='red', linestyle='--', linewidth=1)
_ = axes[0].axvline(x=datetime.strptime('Feb 3 1970 00:00', '%b %d %Y %H:%M'), color='blue', linestyle='--', linewidth=1)
plt.show()


In [ ]:
markets_df.sort_values(by='volume', ascending=False)

In [ ]:
# very liquid tested markets {'market ticker': 'market question}
tested_markets = {'PRES-2024-KH': 'Will Kamala Harris or another Democrat win the Presidency?',
'PRES-2024-DJT': 'Will Donald Trump or another Republican win the Presidency?',
'KXNBA-25-IND': 'Will the Indiana Pacers win the NBA Finals?',
'KXNBA-25-OKC': 'Will the Oklahoma City Thunder win the NBA Finals?',
'POPVOTE-24-D': 'Will the Democratic party win the popular vote?',
'POPVOTE-24-R': 'Will the Republican party win the popular vote?'}

market_vals = {}
trades_record = {}

def ensure_tested_market_history(ticker, period_interval='h'):
    if ticker not in market_vals:
        market_vals[ticker] = load_full_market_cached(
            series_ticker=ticker.split('-')[0],
            ticker=ticker,
            period_interval=period_interval,
        )['candlesticks']
    return market_vals[ticker]

for ticker in tested_markets.keys():
    trades_record[ticker] = routing.get_trades_preview_dataframe_auto(ticker, limit=25)

ensure_tested_market_history('KXNBA-25-IND')


In [ ]:
routing.get_trades_preview_dataframe_auto(ticker='PRES-2024-DJT', limit=25)


In [ ]:
temp = pd.json_normalize(market_vals['KXNBA-25-IND'], sep='_')
from mykalshi import formatting
temp['end_period_ts'] = temp['end_period_ts'].apply(formatting.format_timestamp)
temp

In [ ]:
benchmark_results = []
for r in [5, 10, 20]:
    try:
        result = market.get_all_trades(ticker='PRES-2024-DJT', calls_per_sec=r, batch_size=100)
        ok = result['total_count'] >= 0
    except Exception:
        ok = False
    benchmark_results.append({'calls_per_sec': r, 'trades_success': ok})

pd.DataFrame(benchmark_results)


In [ ]:
market.get_full_market(series_ticker="PRES", ticker="PRES-2024-DJT", start_ts="04/01/2024", end_ts="11/10/2024", period_interval='d')

# market.get_trades(ticker="PRES-2024-DJT", min_ts="04/01/2024", max_ts="11/10/2024")

## 2.0 LOB Tools

In [ ]:
orderbook = market.get_market_orderbook(ticker=LOB_MARKET_TICKER)["orderbook"]
yes_bids = sorted(orderbook["yes"], key=lambda x: x[0])
yes_asks = sorted([[100 - price, size] for price, size in orderbook["no"]], key=lambda x: x[0])

print()
print(f"Ticker: {LOB_MARKET_TICKER}")
if not yes_bids and not yes_asks:
    print("No visible YES-side depth in the current book snapshot.")
print("Bids:")
for price, qty in yes_bids:
    print(f"  YES @ {price}c x {qty:,.2f} contracts")

print("Asks:")
for price, qty in yes_asks:
    print(f"  YES @ {price}c x {qty:,.2f} contracts")


In [ ]:
def get_market_lob(ticker):
    orderbook = market.get_market_orderbook(ticker=ticker)["orderbook"]
    yes_bids = sorted(orderbook["yes"], key=lambda x: x[0])
    yes_asks = sorted([[100 - price, size] for price, size in orderbook["no"]], key=lambda x: x[0])

    print()
    print(f"Ticker: {ticker}")
    if not yes_bids and not yes_asks:
        print("No visible YES-side depth in the current book snapshot.")
        return {"yes_bids": yes_bids, "yes_asks": yes_asks}

    print("Bids:")
    for price, qty in yes_bids:
        print(f"  YES @ {price}c x {qty:,.2f} contracts")

    print("Asks:")
    for price, qty in yes_asks:
        print(f"  YES @ {price}c x {qty:,.2f} contracts")

    return {"yes_bids": yes_bids, "yes_asks": yes_asks}

def plot_market_lob(ticker):
    orderbook = market.get_market_orderbook(ticker=ticker)["orderbook"]
    yes_bids = sorted(orderbook["yes"], key=lambda x: x[0])
    yes_asks = sorted([[100 - price, size] for price, size in orderbook["no"]], key=lambda x: x[0])

    ask_prices = [p for p, _ in yes_asks]
    ask_sizes = [q for _, q in yes_asks]
    ask_cum = list(np.cumsum(ask_sizes)) if ask_sizes else []

    bid_prices = [p for p, _ in yes_bids]
    bid_sizes = [q for _, q in yes_bids]
    bid_cum = list(np.cumsum(bid_sizes[::-1]))[::-1] if bid_sizes else []

    plt.figure(figsize=(10, 6))
    if not bid_prices and not ask_prices:
        plt.text(0.5, 0.5, f"No visible YES-side depth for {ticker}", ha="center", va="center", transform=plt.gca().transAxes)
        plt.xlim(0, 100)
        plt.ylim(0, 1)
        plt.xlabel("Price (c)")
        plt.ylabel("Cumulative Size")
        plt.title("YES Order Book Depth")
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        return
    if bid_prices:
        bid_prices_ext = bid_prices + [bid_prices[-1]]
        bid_cum_ext = bid_cum + [0]
        plt.step(bid_prices_ext, bid_cum_ext, label="Bids", color="green", where="post")
        plt.fill_between(bid_prices_ext, bid_cum_ext, step="post", color="green", alpha=0.3, hatch='//')
    if ask_prices:
        ask_prices_ext = [ask_prices[0]] + ask_prices
        ask_cum_ext = [0] + ask_cum
        plt.step(ask_prices_ext, ask_cum_ext, label="Asks", color="red", where="post")
        plt.fill_between(ask_prices_ext, ask_cum_ext, step="post", color="red", alpha=0.3, hatch='\\')

    plt.xlabel("Price (c)")
    plt.ylabel("Cumulative Size")
    plt.title("YES Order Book Depth")
    plt.legend(loc="upper center")
    plt.xlim(0, 100)
    plt.grid(True)

    max_val = max(max(bid_cum, default=0), max(ask_cum, default=0))
    if max_val >= 1_000_000:
        divisor = 1_000_000
        suffix = "M"
    elif max_val >= 1_000:
        divisor = 1_000
        suffix = "K"
    else:
        divisor = 1
        suffix = ""

    plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x / divisor:.1f}{suffix}"))
    plt.tight_layout()
    plt.show()


In [ ]:
get_market_lob(LOB_MARKET_TICKER)


In [ ]:
plot_market_lob(LOB_MARKET_TICKER)


In [ ]:
import time

def live_market_lob(ticker, refresh_interval=1, iterations=5):
    try:
        for _ in range(iterations):
            loop_start = time.time()
            os.system('cls' if os.name == 'nt' else 'clear')
            get_market_lob(ticker)
            elapsed = time.time() - loop_start
            time.sleep(max(0, refresh_interval - elapsed))
    except KeyboardInterrupt:
        print()
        print("Stopped.")


In [ ]:
import time
from IPython.display import clear_output

def live_market_lob_notebook(ticker, refresh_interval=1, iterations=5):
    try:
        for _ in range(iterations):
            loop_start = time.time()
            clear_output(wait=True)
            get_market_lob(ticker)
            elapsed = time.time() - loop_start
            time.sleep(max(0, refresh_interval - elapsed))
    except KeyboardInterrupt:
        print()
        print("Stopped.")


In [ ]:
import time
from IPython.display import clear_output

def live_plot_market_lob(ticker, refresh_interval=1, iterations=5):
    try:
        for _ in range(iterations):
            loop_start = time.time()
            clear_output(wait=True)
            plot_market_lob(ticker)
            elapsed = time.time() - loop_start
            time.sleep(max(0, refresh_interval - elapsed))
    except KeyboardInterrupt:
        clear_output(wait=True)
        print("Live plot stopped.")


In [ ]:
routing.get_trades_preview_dataframe_auto(LOB_MARKET_TICKER, limit=25)


In [ ]:
live_market_lob_notebook(LOB_MARKET_TICKER, refresh_interval=2, iterations=3)


In [ ]:
live_plot_market_lob(LOB_MARKET_TICKER, refresh_interval=2, iterations=3)


In [ ]:
# def record_orderbook_time_series(ticker, duration_secs=60, interval_secs=1.0):
#     import time
#     from datetime import datetime

#     snapshots = []
#     snapshot_count = 0
#     start_time = time.time()

#     try:
#         while time.time() - start_time < duration_secs:
#             loop_start = time.time()

#             # Fetch and parse orderbook
#             orderbook = market.get_market_orderbook(ticker=ticker)["orderbook"]
#             yes_bids = orderbook["yes"]
#             yes_asks = [[100 - price, size] for price, size in orderbook["no"]]

#             bid_snapshot = {int(price): int(size) for price, size in yes_bids if size > 0}
#             ask_snapshot = {int(price): int(size) for price, size in yes_asks if size > 0}

#             now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

#             snapshots.append({
#                 "timestamp": now,
#                 "bids": bid_snapshot,
#                 "asks": ask_snapshot
#             })

#             snapshot_count += 1

#             elapsed = time.time() - loop_start
#             time.sleep(max(0, interval_secs - elapsed))

#     except KeyboardInterrupt:
#         print("Stopped by user.")

#     import sys
#     from sys import getsizeof

#     total_bytes = sum(getsizeof(s) for s in snapshots)
#     size_mb = total_bytes / (1024 ** 2)

#     print(f"\nRecording complete: {snapshot_count} snapshots collected.")
#     print(f"Total memory used: {size_mb:.2f} MB")

#     return snapshots


In [ ]:
# snapshot = record_orderbook_time_series("KXFEDDECISION-25JUL-C25", duration_secs=60, interval_secs=1)

In [ ]:
# import pandas as pd

# def snapshot_list_to_dfs(snapshots):
#     price_levels = list(range(101))
#     bids_ts = pd.DataFrame(columns=price_levels)
#     asks_ts = pd.DataFrame(columns=price_levels)

#     for snap in snapshots:
#         timestamp = snap["timestamp"]
#         bids_row = {int(p): snap["bids"].get(p, 0) for p in price_levels}
#         asks_row = {int(p): snap["asks"].get(p, 0) for p in price_levels}
#         bids_ts.loc[timestamp] = bids_row
#         asks_ts.loc[timestamp] = asks_row

#     return bids_ts.astype(int), asks_ts.astype(int)

In [ ]:
# # Convert back to DataFrames
# bids_df, asks_df = snapshot_list_to_dfs(snapshot)

# asks_df

In [ ]:
# open_mkts.sort_values(by='volume', ascending=False)

In [ ]:
import time
import random
import threading
import queue
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import HTTPError

class MarketLOBRecorder:
    def __init__(self,
                 tickers,
                 interval_secs: float = 0.5,
                 max_workers: int = None,
                 max_retries: int = 5,
                 base_backoff: float = 0.1,
                 calls_per_sec: int = 30,
                 output_path: str = "lob_stream.jsonl"):
        self.tickers = tickers
        self.interval_secs = interval_secs
        self.max_retries = max_retries
        self.base_backoff = base_backoff

        # sensible default for max_workers
        self.max_workers = max_workers if max_workers is not None else min(32, len(tickers))
        self._executor = ThreadPoolExecutor(self.max_workers)

        # rate-limiter setup based on API tier
        self.min_interval = 1.0 / calls_per_sec
        self._lock = threading.Lock()
        self._last_call = 0.0

        # error tracking
        self.error_counts = {tk: 0 for tk in tickers}

        # streaming writer setup
        self._write_q = queue.Queue(maxsize=10000)
        self._out_fh = open(output_path, "w")
        self._writer_thread = threading.Thread(target=self._writer_loop, daemon=True)
        self._writer_thread.start()

    def _writer_loop(self):
        """Continuously write JSONL records from the queue to disk."""
        while True:
            rec = self._write_q.get()
            if rec is None:
                break
            self._out_fh.write(json.dumps(rec) + "\n")
            # flush occasionally for safety
            if self._write_q.qsize() < 100:
                self._out_fh.flush()
        self._out_fh.flush()
        self._out_fh.close()

    def _wait_rate_limit(self):
        """Enforce global calls_per_sec limit across threads."""
        with self._lock:
            now = time.time()
            elapsed = now - self._last_call
            if elapsed < self.min_interval:
                time.sleep(self.min_interval - elapsed)
            self._last_call = time.time()

    def _fetch_one(self, ticker):
        """Fetch a single LOB with retries, backoff, and rate-limiting."""
        last_exc = None
        for attempt in range(1, self.max_retries + 1):
            try:
                self._wait_rate_limit()
                resp = market.get_market_orderbook(ticker=ticker)
                book = resp.get("orderbook") or {}
                yes_list = book.get("yes") if isinstance(book.get("yes"), list) else []
                no_list = book.get("no") if isinstance(book.get("no"), list) else []

                bids = {int(p): int(sz) for p, sz in yes_list if sz > 0}
                asks = {int(100-p): int(sz) for p, sz in no_list if sz > 0}

                if not yes_list and not no_list:
                    raise ValueError("Empty orderbook arrays")

                record = {
                    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
                    "ticker": ticker,
                    "bids": bids,
                    "asks": asks
                }
                return record

            except HTTPError as http_err:
                status = getattr(http_err.response, "status_code", None)
                if status == 429 and attempt < self.max_retries:
                    # exponential backoff + jitter
                    delay = self.base_backoff * (2 ** (attempt - 1))
                    delay *= random.uniform(0.8, 1.2)
                    time.sleep(delay)
                    last_exc = http_err
                    continue
                last_exc = http_err
                break

            except Exception as exc:
                # jitter on other errors
                delay = self.base_backoff * random.uniform(0.5, 1.5)
                time.sleep(delay)
                last_exc = exc
                continue

        # all retries exhausted or forced error
        self.error_counts[ticker] += 1
        record = {
            "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
            "ticker": ticker,
            "bids": {},
            "asks": {},
            "error": repr(last_exc)
        }
        return record

    def _fetch_all(self):
        """Fetch LOBs for all tickers in parallel and stream records to disk."""
        futures = {self._executor.submit(self._fetch_one, tk): tk for tk in self.tickers}
        records = [f.result() for f in as_completed(futures)]
        for rec in records:
            self._write_q.put(rec)
        return records

    def start(self, duration_secs: float):
        """Run the polling loop and stream records until duration_secs elapses."""
        end = time.time() + duration_secs
        while time.time() < end:
            cycle_start = time.time()
            self._fetch_all()
            elapsed = time.time() - cycle_start
            time.sleep(max(0, self.interval_secs - elapsed))

        # signal writer thread to shut down
        self._write_q.put(None)
        self._writer_thread.join()
        print("Done streaming to disk.")
        print("Errors by ticker:", self.error_counts)


In [ ]:
# import os
# import json
# import time
# import numpy as np
# import pandas as pd

# # ————— PARAMETERS TO SWEEP —————
# ticker_counts = [100, 200]  # number of tickers to record
# intervals    = [5.0, 10.0]  # seconds between cycles
# worker_counts= [22, 44]  # number of threads to use

# duration_secs = 40  # keep runs short for testing

# ALL_TICKERS = open_mkts.sort_values(by='volume', ascending=False)\
#                       .head(200)['ticker'].tolist()

# results = []

# def dump_snapshots_to_jsonl(snaps, fname):
#     with open(fname, "w") as f:
#         for s in snaps:
#             f.write(json.dumps(s) + "\n")
#     return os.path.getsize(fname)

# for n in ticker_counts:
#     tickers = ALL_TICKERS[:n]
#     for interval in intervals:
#         seen_workers = set()
#         for workers in worker_counts:
#             max_workers = min(workers, n)
#             if max_workers in seen_workers:
#                 continue
#             seen_workers.add(max_workers)

#             # initialize and run recorder
#             rec = MarketLOBRecorder(
#                 tickers=tickers,
#                 interval_secs=interval,
#                 max_workers=max_workers
#             )
#             t0 = time.time()
#             rec.start(duration_secs=duration_secs)
#             total_time = time.time() - t0

#             # count errors
#             error_count = sum(rec.error_counts.values())
#             total_snaps = len(rec.snapshots)
#             error_rate  = error_count / total_snaps if total_snaps else float('nan')

#             # dump to JSONL and measure size
#             jsonl_file = f"test_n{n}_int{interval}_w{max_workers}.jsonl"
#             size_bytes = dump_snapshots_to_jsonl(rec.snapshots, jsonl_file)

#             # build DataFrame of timestamps
#             df_snaps = pd.DataFrame(rec.snapshots)
#             df_snaps['ts_dt'] = pd.to_datetime(
#                 df_snaps['timestamp'], format='%Y-%m-%d %H:%M:%S.%f'
#             )

#             # compute per-ticker MSE of observed intervals vs target
#             mse_list = []
#             for _, group in df_snaps.groupby('ticker'):
#                 times  = np.sort(group['ts_dt'].values)
#                 deltas = np.diff(times).astype('timedelta64[ns]').astype(np.float64) / 1e9
#                 errs   = (deltas - interval) ** 2
#                 if len(errs):
#                     mse_list.append(errs.mean())
#             interval_mse = float(np.mean(mse_list)) if mse_list else float('nan')

#             # compute other timing metrics
#             cycles      = total_snaps / n if n else 0
#             avg_cycle   = total_time / cycles if cycles else float("nan")
#             avg_latency = avg_cycle / n if n else float("nan")

#             results.append({
#                 "n_tickers":     n,
#                 "interval_s":    interval,
#                 "max_workers":   max_workers,
#                 "calls_per_sec": rec.calls_per_sec if hasattr(rec, 'calls_per_sec') else None,
#                 "avg_cycle_s":   round(avg_cycle, 3),
#                 "avg_latency_s": round(avg_latency, 3),
#                 "file_size_MB":  round(size_bytes / (1024**2), 3),
#                 "interval_mse":  round(interval_mse,   6),
#                 "error_count":   error_count,
#                 "error_rate":    round(error_rate,    3),
#             })

#             # cleanup
#             os.remove(jsonl_file)
#             print(f"Done n={n}, int={interval}, w={max_workers}, calls={calls} ")

# # summarize in a DataFrame
# df = pd.DataFrame(results)
# df

In [ ]:
# open_mkts.sort_values(by='volume', ascending=False).head(500).sort_values(by='close_time', ascending=True)
open_mkts.sort_values(by='close_time', ascending=True).head(3000).sort_values(by='volume', ascending=False)

In [ ]:
import time
from datetime import datetime
import numpy as np
import pandas as pd

capture_path = Path(tempfile.gettempdir()) / f"{LOB_MARKET_TICKER.replace('-', '_')}_lob_snapshots.jsonl"
rec = MarketLOBRecorder(
    tickers=[LOB_MARKET_TICKER],
    interval_secs=2,
    max_workers=1,
    calls_per_sec=5,
    output_path=str(capture_path),
)

t0 = time.time()
rec.start(duration_secs=12)
total_duration = time.time() - t0

df = pd.read_json(capture_path, lines=True)
df['ts_dt'] = pd.to_datetime(df['timestamp'])


In [ ]:
total_records = len(df)
error_count = df['error'].notnull().sum() if 'error' in df.columns else 0
error_rate = error_count / total_records if total_records else float('nan')

mse_list = []
all_deltas = []

for tk, group in df.groupby('ticker'):
    times = group.sort_values('ts_dt')['ts_dt']
    deltas = times.diff().dt.total_seconds().dropna()
    if not deltas.empty:
        mse_list.append(((deltas - 2.0)**2).mean())
        all_deltas.extend(deltas.values)

precision_mse = float(np.mean(mse_list)) if mse_list else float('nan')
mean_cycle = float(np.mean(all_deltas)) if all_deltas else float('nan')
std_cycle = float(np.std(all_deltas)) if all_deltas else float('nan')

print(f"Total duration:         {total_duration:.2f} s")
print(f"Total records:          {total_records}")
print(f"Error count:            {error_count}")
print(f"Error rate:             {error_rate:.2%}")
print(f"Precision MSE (2s):     {precision_mse:.6f}")
print(f"Mean cycle length:      {mean_cycle:.3f} s")
print(f"Cycle length std dev:   {std_cycle:.3f} s")


In [ ]:
open_mkts[open_mkts['volume'] > 30000].sort_values(by='close_time', ascending=True)

In [ ]:
# open df from aws s3 bucket
import pandas as pd
import boto3
import io

def load_s3_jsonl(bucket_name, folder_key, file_key):
    s3 = boto3.client('s3')
    # Construct the full S3 key
    full_key = f"{folder_key}/{file_key}" if folder_key else file_key
    # Fetch the object from S3
    response = s3.get_object(Bucket=bucket_name, Key=full_key)
    data = response['Body'].read().decode('utf-8')
    
    # Read the JSONL data into a DataFrame
    df = pd.read_json(io.StringIO(data), lines=True)
    return df

# Example usage
bucket_name = 'mykalshi-lob-logs'
folder_key = 'daily'
file_keys = ['lob_stream_20250717.jsonl', 'lob_stream_20250718.jsonl', 'lob_stream_20250719.jsonl', 'lob_stream_20250720.jsonl', 'lob_stream_20250721.jsonl']

dfs = []

for file_key in file_keys:
    df = load_s3_jsonl(bucket_name, folder_key, file_key)
    dfs.append(df)

# Combine all DataFrames into one
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp'], format='%Y-%m-%d %H:%M:%S.%f')
combined_df_clean = combined_df[combined_df['error'].isnull()]
combined_df_clean

In [ ]:
# chewck start and end timestamps for each df in dfs
for i, df in enumerate(dfs):
    start_ts = df['timestamp'].min()
    end_ts = df['timestamp'].max()
    print(f"File {file_keys[i]}: Start TS: {start_ts}, End TS: {end_ts}")

In [ ]:
import pandas as pd

def snapshot_list_to_dfs(snapshots):
    price_levels = list(range(1, 100))
    bids_data = []
    asks_data = []
    timestamps = []

    for snap in snapshots:
        if snap.get('error'):
            continue
        timestamps.append(pd.to_datetime(snap["timestamp"], utc=True))
        bids_data.append([snap["bids"].get(str(p), snap["bids"].get(p, 0)) for p in price_levels])
        asks_data.append([snap["asks"].get(str(p), snap["asks"].get(p, 0)) for p in price_levels])

    bids_ts = pd.DataFrame(bids_data, index=timestamps, columns=price_levels).astype(float)
    asks_ts = pd.DataFrame(asks_data, index=timestamps, columns=price_levels).astype(float)
    return bids_ts, asks_ts

filtered_snapshots = df[df['ticker'] == LOB_MARKET_TICKER].to_dict(orient="records")
bids_df, asks_df = snapshot_list_to_dfs(filtered_snapshots)
asks_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker

def plot_lob_at_index(idx, bids_ts, asks_ts):
    snapshot_time = bids_ts.index[idx]
    bids = bids_ts.iloc[idx]
    asks = asks_ts.iloc[idx]

    yes_bids = sorted([[p, bids[p]] for p in bids.index if bids[p] > 0], key=lambda x: x[0])
    yes_asks = sorted([[p, asks[p]] for p in asks.index if asks[p] > 0], key=lambda x: x[0])

    bid_prices = [p for p, _ in yes_bids]
    bid_sizes = [q for _, q in yes_bids]
    bid_cum = list(np.cumsum(bid_sizes[::-1]))[::-1] if bid_sizes else []

    ask_prices = [p for p, _ in yes_asks]
    ask_sizes = [q for _, q in yes_asks]
    ask_cum = list(np.cumsum(ask_sizes)) if ask_sizes else []

    plt.figure(figsize=(11, 5))

    if yes_bids:
        bid_prices_ext = bid_prices + [bid_prices[-1]]
        bid_cum_ext = bid_cum + [0]
        plt.step(bid_prices_ext, bid_cum_ext, label="Bids", color="green", where="post")
        plt.fill_between(bid_prices_ext, bid_cum_ext, step="post", color="green", alpha=0.3, hatch='//')

    if yes_asks:
        ask_prices_ext = [ask_prices[0]] + ask_prices
        ask_cum_ext = [0] + ask_cum
        plt.step(ask_prices_ext, ask_cum_ext, label="Asks", color="red", where="post")
        plt.fill_between(ask_prices_ext, ask_cum_ext, step="post", color="red", alpha=0.3, hatch='\\')

    plt.title(f"LOB at {snapshot_time}")
    plt.xlabel("Price (c)")
    plt.ylabel("Cumulative Size")
    plt.xlim(0, 100)
    plt.grid(True)
    plt.legend(loc="upper center")

    max_val = max(max(bid_cum, default=0), max(ask_cum, default=0))
    if max_val >= 1_000_000:
        divisor, suffix = 1_000_000, "M"
    elif max_val >= 1_000:
        divisor, suffix = 1_000, "K"
    else:
        divisor, suffix = 1, ""

    plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x / divisor:.1f}{suffix}"))
    plt.tight_layout()
    plt.show()

if IPYWIDGETS_AVAILABLE and len(bids_df) > 0:
    interact(
        plot_lob_at_index,
        idx=IntSlider(min=0, max=len(bids_df) - 1, step=1, value=0),
        bids_ts=fixed(bids_df),
        asks_ts=fixed(asks_df)
    )
elif len(bids_df) > 0:
    plot_lob_at_index(0, bids_df=bids_df, asks_ts=asks_df)
else:
    print("No clean order book snapshots were captured.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

order_size = 10
inventory = 0
cash = 0
inventory_path = []
cash_path = []
midprice_path = []
timestamp_path = []

timestamps = bids_df.index.to_list()

for i in range(1, len(timestamps)):
    t_prev = timestamps[i - 1]
    t_curr = timestamps[i]
    bid_prev = bids_df.loc[t_prev]
    ask_prev = asks_df.loc[t_prev]
    bid_curr = bids_df.loc[t_curr]
    ask_curr = asks_df.loc[t_curr]

    bid_prices = [p for p in bid_prev.index if bid_prev[p] > 0]
    ask_prices = [p for p in ask_prev.index if ask_prev[p] > 0]
    if not bid_prices or not ask_prices:
        continue

    best_bid = max(bid_prices)
    best_ask = min(ask_prices)
    mid = 0.5 * (best_bid + best_ask)

    bid_filled = best_bid in bid_curr and (bid_prev[best_bid] - bid_curr[best_bid]) >= order_size
    ask_filled = best_ask in ask_curr and (ask_prev[best_ask] - ask_curr[best_ask]) >= order_size

    if bid_filled:
        inventory += order_size
        cash -= order_size * best_bid
    if ask_filled:
        inventory -= order_size
        cash += order_size * best_ask

    inventory_path.append(inventory)
    cash_path.append(cash)
    midprice_path.append(mid)
    timestamp_path.append(t_curr)

pnl = pd.Series(cash_path) + pd.Series(inventory_path) * pd.Series(midprice_path)

plt.figure(figsize=(12, 6))
palette = sns.color_palette("tab10")
plt.plot(timestamp_path, pnl, label="PnL", color=palette[0])
plt.ylabel("PnL")
plt.legend(loc="upper left")
plt.twinx().plot(timestamp_path, inventory_path, label="Inventory", color=palette[1])
plt.ylabel("Inventory (contracts)")
plt.xticks(rotation=45)
plt.title("FIFO-Fill Simulated Passive MM")
plt.xlabel("Time")
plt.tight_layout()
plt.show()


In [ ]:
family_markets = normalize_market_frame(
    pd.json_normalize(market.get_markets(series_ticker='KXHIGHMIA', status='open', limit=10)['markets'])
).sort_values('volume', ascending=False)
family_tickers = family_markets['ticker'].head(4).tolist()
family_capture_path = Path(tempfile.gettempdir()) / 'mykalshi_weather_family_lob.jsonl'
family_rec = MarketLOBRecorder(
    tickers=family_tickers,
    interval_secs=4,
    max_workers=min(4, len(family_tickers)),
    calls_per_sec=8,
    output_path=str(family_capture_path),
)
family_rec.start(duration_secs=12)
family_df = pd.read_json(family_capture_path, lines=True)
family_df


In [ ]:
results = {}
for ticker in family_tickers:
    snapshots = family_df[family_df['ticker'] == ticker].to_dict(orient="records")
    if len(snapshots) < 2:
        continue
    bids_df, asks_df = snapshot_list_to_dfs(snapshots)

    order_size = 10
    inventory = 0
    cash = 0
    inventory_path = []
    cash_path = []
    midprice_path = []
    timestamp_path = []

    timestamps = bids_df.index.to_list()
    for i in range(1, len(timestamps)):
        t_prev = timestamps[i - 1]
        t_curr = timestamps[i]
        bid_prev = bids_df.loc[t_prev]
        ask_prev = asks_df.loc[t_prev]
        bid_curr = bids_df.loc[t_curr]
        ask_curr = asks_df.loc[t_curr]
        bid_prices = [p for p in bid_prev.index if bid_prev[p] > 0]
        ask_prices = [p for p in ask_prev.index if ask_prev[p] > 0]
        if not bid_prices or not ask_prices:
            continue
        best_bid = max(bid_prices)
        best_ask = min(ask_prices)
        mid = 0.5 * (best_bid + best_ask)
        bid_filled = best_bid in bid_curr and (bid_prev[best_bid] - bid_curr[best_bid]) >= order_size
        ask_filled = best_ask in ask_curr and (ask_prev[best_ask] - ask_curr[best_ask]) >= order_size
        if bid_filled:
            inventory += order_size
            cash -= order_size * best_bid
        if ask_filled:
            inventory -= order_size
            cash += order_size * best_ask
        inventory_path.append(inventory)
        cash_path.append(cash)
        midprice_path.append(mid)
        timestamp_path.append(t_curr)

    if timestamp_path:
        pnl = pd.Series(cash_path) + pd.Series(inventory_path) * pd.Series(midprice_path)
        results[ticker] = {"timestamp": timestamp_path, "pnl": pnl, "inventory": inventory_path}

if results:
    n = len(results)
    cols = 2
    rows = (n + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(14, 3.5 * rows), sharex=False)
    axs = axs.flatten()
    for i, (ticker, data) in enumerate(results.items()):
        ax = axs[i]
        ax2 = ax.twinx()
        ax.plot(data["timestamp"], data["pnl"], label="PnL", color="tab:blue")
        ax2.plot(data["timestamp"], data["inventory"], label="Inventory", color="tab:orange")
        ax.set_title(ticker)
        ax.set_ylabel("PnL")
        ax2.set_ylabel("Inventory")
        ax.tick_params(axis='x', rotation=45)
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])
    fig.suptitle("Simulated FIFO-Fill MM Performance Across Markets", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
else:
    print("The short family capture did not collect enough movement for the multi-market simulation grid.")


In [ ]:
pd.DataFrame({"ticker": family_tickers})


In [ ]:
pd.DataFrame({"ticker": list(results.keys())}) if results else pd.DataFrame()


In [ ]:
for tkr, r in results.items():
    print(f"{tkr:20s} Last simulated PnL: {r['pnl'].iloc[-1]:.2f}")


In [ ]:
for tkr, r in results.items():
    print(f"{tkr:20s} Final PnL: ${r['final_pnl']:.2f}")


## 3.0 Correlation Analysis

In [ ]:
weather_series = pd.json_normalize(events.get_series_list(category='Climate and Weather')['series'])
weather_titles = pd.Series(weather_series['title'].unique())
weather_titles.sort_values()
weather_series


In [ ]:
weather_series_dict = WEATHER_SERIES.copy()
weather_series_dict


In [ ]:
events.get_series(series_ticker='KXHIGHAUS')

In [ ]:
weather_events = pd.DataFrame()
for series_ticker in weather_series_dict.keys():
    events_temp = pd.json_normalize(
        retry_on_429(
            f"weather events {series_ticker}",
            lambda st=series_ticker: events.get_events(series_ticker=st, limit=25),
        )['events']
    )
    weather_events = pd.concat([weather_events, events_temp], ignore_index=True)
weather_events['City'] = extract_city_from_text(weather_events['title'])
weather_events


In [ ]:
weather_markets = pd.DataFrame()
for series_ticker in weather_series_dict.keys():
    events_temp = pd.json_normalize(
        retry_on_429(
            f"weather markets {series_ticker}",
            lambda st=series_ticker: market.get_markets(series_ticker=st, status='open', limit=25),
        )['markets']
    )
    weather_markets = pd.concat([weather_markets, events_temp], ignore_index=True)
weather_markets = normalize_market_frame(weather_markets)
weather_markets['City'] = extract_city_from_text(weather_markets['yes_sub_title'])
weather_markets


In [ ]:
weather_events

# 4.0 IMDB

In [ ]:
try:
    import requests
    from bs4 import BeautifulSoup
except ModuleNotFoundError:
    print("Install beautifulsoup4 to run the legacy IMDB appendix.")


In [ ]:
if 'BeautifulSoup' in globals():
    def get_rating_distribution(title_id):
        url = f"https://www.imdb.com/title/{title_id}/ratings"
        headers = {'User-Agent': 'Mozilla/5.0'}
        res = requests.get(url, headers=headers, timeout=15)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, 'html.parser')
        dist = {}
        for row in soup.select('table.imdbRatingTable tr'):
            cols = row.find_all('td')
            if len(cols) == 3:
                rating = int(cols[0].text.strip())
                count = int(cols[2].text.strip().replace(',', ''))
                dist[rating] = count
        return dist


In [ ]:
if 'get_rating_distribution' in globals():
    dist = get_rating_distribution('tt0133093')
    dist


In [ ]:
if 'BeautifulSoup' in globals():
    url = f"https://www.imdb.com/title/tt0133093/ratings"
    headers = {'User-Agent': 'Mozilla/5.0'}
    res = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')
    soup
